In [ ]:
brew install pinocchio


Channels:
 - conda-forge
 - defaults
Platform: osx-64
Solving environment: done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [1]:
#create a urdf file and verify your installation
import pinocchio
print(dir(pinocchio))





['ACCELERATION', 'ADMMContactSolver', 'ARG0', 'ARG1', 'ARG2', 'ARG3', 'ARG4', 'AngleAxis', 'ArgumentPosition', 'BODY', 'BaumgarteCorrectorParameters', 'BroadPhaseManagerPool_DynamicAABBTreeCollisionManager', 'BroadPhaseManager_DynamicAABBTreeArrayCollisionManager', 'BroadPhaseManager_DynamicAABBTreeCollisionManager', 'BroadPhaseManager_IntervalTreeCollisionManager', 'BroadPhaseManager_NaiveCollisionManager', 'BroadPhaseManager_SSaPCollisionManager', 'BroadPhaseManager_SaPCollisionManager', 'COLLISION', 'CachedMeshLoader', 'CollisionCallBackBase', 'CollisionCallBackDefault', 'CollisionGeometry', 'CollisionObject', 'CollisionPair', 'CollisionResult', 'ComputeCollision', 'ComputeDistance', 'Contact', 'ContactCholeskyDecomposition', 'ContactType', 'Convention', 'CoulombFrictionCone', 'Data', 'DelassusCholeskyExpression', 'DelassusOperatorDense', 'DelassusOperatorSparse', 'DeprecatedWarning', 'DistanceResult', 'DualCoulombFrictionCone', 'Exception', 'FIXED_JOINT', 'Force', 'Frame', 'FrameTy

In [2]:
pip install numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [6]:
#Run this script to be sure 2-link robot arm is working perfectly
import pinocchio as pin
import numpy as np

# Replace with your robot's URDF path
urdf_path = '/Users/user_1/mlpro/HALA/robot_arm_2link.urdf'

# Load robot model
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()

# Get DOF counts
nq = model.nq
nv = model.nv

# Generate random joint configs within pinocchio position/velocity limits
q_lower = model.lowerPositionLimit
q_upper = model.upperPositionLimit
qd_limit = model.velocityLimit

q_samples = np.random.uniform(q_lower, q_upper, size=(1000, nq))
qd_samples = np.random.uniform(-qd_limit, qd_limit, size=(1000, nv))  # velocity

# Dynamics computation per sample
results = []
for q, qd in zip(q_samples, qd_samples):
    pin.computeAllTerms(model, data, q, qd)
    M = data.M
    Cqd = data.nle - data.g          # non-linear effects minus gravity yields C(q,q̇)q̇
    g_vec = data.g
    tau = np.ones(nv)                # constant torque, can be customized

    b = tau - Cqd - g_vec

    # Baseline solution for reference
    qddot_ref = np.linalg.solve(M, b)

    # Implement algorithm variants: Gauss-Jordan, Neumann, SPAI, HALA here as needed...
    # Store timing and error for each run
    results.append({
        'M': M,
        'b': b,
        'qddot_ref': qddot_ref
        # Add HALA or other algorithm outputs and metrics here
    })


In [12]:
import pinocchio as pin
import numpy as np
import time

# Algorithm Implementations (as above)
def gauss_jordan(M, b):
    return np.linalg.solve(M, b)

def neumann_series_inverse(M, num_terms=10):
    M0 = np.diag(np.diag(M))
    M0_inv = np.linalg.inv(M0)
    E = np.eye(M.shape[0]) - M0_inv @ M
    S = np.eye(M.shape[0])
    term = np.eye(M.shape[0])
    for k in range(1, num_terms):
        term = term @ E
        S = S + term
    return S @ M0_inv

def spai_inverse(M):
    return np.diag(1 / np.diag(M))

def hala(M, b, num_neumann=5):
    G_spai = spai_inverse(M)
    E = np.eye(M.shape[0]) - G_spai @ M
    S = np.eye(M.shape[0])
    term = np.eye(M.shape[0])
    for k in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    qddot_hala = M_inv_hala @ b
    error = np.linalg.norm(M @ qddot_hala - b)
    if error > 1e-3:
        qddot_hala = gauss_jordan(M, b)
    return qddot_hala

# Benchmark Script With Stability Check
urdf_path = '/Users/user_1/mlpro/HALA/ur5robot.urdf'
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq = model.nq
nv = model.nv
q_lower = model.lowerPositionLimit
q_upper = model.upperPositionLimit
qd_limit = model.velocityLimit

# ---- Fix begins here ----
# Replace inf/nan/large values in qd_limit with a reasonable max velocity (e.g., 1.0, or clamp to 10)
qd_limit_safe = np.copy(qd_limit)
qd_limit_safe[~np.isfinite(qd_limit_safe)] = 1.0  # Replace inf or nan with 1.0
qd_limit_safe = np.clip(qd_limit_safe, 0, 10)
# ---- Fix ends here ----

num_runs = 1000
q_samples = np.random.uniform(q_lower, q_upper, size=(num_runs, nq))
qd_samples = np.random.uniform(-qd_limit_safe, qd_limit_safe, size=(num_runs, nv))

results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'neumann_error': [], 'spai_error': [], 'hala_error': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

for q, qd in zip(q_samples, qd_samples):
    pin.computeAllTerms(model, data, q, qd)
    M = data.M
    Cqd = data.nle - data.g
    g_vec = data.g
    tau = np.ones(nv)
    b = tau - Cqd - g_vec

    # Perturbation for stability
    delta_M = np.random.randn(*M.shape) * 1e-6
    M_pert = M + delta_M

    # Reference
    t0 = time.time()
    qddot_ref = gauss_jordan(M, b)
    t1 = time.time()
    results['ref_time'].append((t1 - t0) * 1000)
    qddot_ref_pert = gauss_jordan(M_pert, b)
    kappa_ref = (np.linalg.norm(qddot_ref_pert - qddot_ref) / np.linalg.norm(qddot_ref)) / \
                (np.linalg.norm(delta_M) / np.linalg.norm(M))
    results['ref_kappa'].append(kappa_ref)

    # Neumann
    t0 = time.time()
    neumann_inv = neumann_series_inverse(M) 
    qddot_neumann = neumann_inv @ b
    t1 = time.time()
    results['neumann_time'].append((t1 - t0) * 1000)
    results['neumann_error'].append(np.linalg.norm(qddot_neumann - qddot_ref) / np.linalg.norm(qddot_ref))
    qddot_neumann_pert = neumann_series_inverse(M_pert) @ b
    kappa_neumann = (np.linalg.norm(qddot_neumann_pert - qddot_ref) / np.linalg.norm(qddot_ref)) / \
                    (np.linalg.norm(delta_M) / np.linalg.norm(M))
    results['neumann_kappa'].append(kappa_neumann)

    # SPAI
    t0 = time.time()
    spai_inv = spai_inverse(M)
    qddot_spai = spai_inv @ b
    t1 = time.time()
    results['spai_time'].append((t1 - t0) * 1000)
    results['spai_error'].append(np.linalg.norm(qddot_spai - qddot_ref) / np.linalg.norm(qddot_ref))
    spai_inv_pert = spai_inverse(M_pert)
    qddot_spai_pert = spai_inv_pert @ b
    kappa_spai = (np.linalg.norm(qddot_spai_pert - qddot_ref) / np.linalg.norm(qddot_ref)) / \
                 (np.linalg.norm(delta_M) / np.linalg.norm(M))
    results['spai_kappa'].append(kappa_spai)

    # HALA
    t0 = time.time()
    qddot_hala = hala(M, b, num_neumann=10)
    t1 = time.time()
    results['hala_time'].append((t1 - t0) * 1000)
    results['hala_error'].append(np.linalg.norm(qddot_hala - qddot_ref) / np.linalg.norm(qddot_ref))
    qddot_hala_pert = hala(M_pert, b, num_neumann=10)
    kappa_hala = (np.linalg.norm(qddot_hala_pert - qddot_ref) / np.linalg.norm(qddot_ref)) / \
                 (np.linalg.norm(delta_M) / np.linalg.norm(M))
    results['hala_kappa'].append(kappa_hala)

# ---------- Summarize Results ----------
def summarize(name, times, errors, kappas):
    print(f"{name}: Avg Time (ms): {np.mean(times):.2f}, Avg Error: {np.mean(errors):.2e}, Avg Stability (kappa): {np.mean(kappas):.2f}")

summarize("Gauss-Jordan (ref)", results['ref_time'], [0]*num_runs, results['ref_kappa'])
summarize("Neumann Series", results['neumann_time'], results['neumann_error'], results['neumann_kappa'])
summarize("SPAI", results['spai_time'], results['spai_error'], results['spai_kappa'])
summarize("HALA", results['hala_time'], results['hala_error'], results['hala_kappa'])


Gauss-Jordan (ref): Avg Time (ms): 0.02, Avg Error: 0.00e+00, Avg Stability (kappa): 27.13
Neumann Series: Avg Time (ms): 0.07, Avg Error: 4.42e-01, Avg Stability (kappa): 326830.19
SPAI: Avg Time (ms): 0.01, Avg Error: 4.16e-01, Avg Stability (kappa): 250286.60
HALA: Avg Time (ms): 0.07, Avg Error: 0.00e+00, Avg Stability (kappa): 27.13
